# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table

In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id)

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"
print(bronze_table)
print(silver_table)

In [0]:
circuits_df = spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)

In [0]:
circuits_drop_df = circuits_df.select(
    "circuitId",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "Ingestion_Timestamp",
    "Source_File",
    "batch_id",
)

In [0]:
circuits_renamed_col_df = circuits_drop_df.withColumnsRenamed(
    {
        "circuitId": "circuit_id",
        "circuitName": "circuit_name",
        "lat": "latitude",
        "long": "longitude",
    }
)

In [0]:
circuits_drop_null_df = circuits_renamed_col_df.na.drop(subset=["circuit_id"])

In [0]:
circuits_drop_duplicates_df = circuits_drop_null_df.dropDuplicates(["circuit_id"])

In [0]:
circuits_col_title_cap = circuits_drop_duplicates_df.withColumns(
    {
        "circuit_id": F.initcap(F.col("circuit_id")),
        "locality": F.initcap(F.col("locality")),
    }
)

In [0]:
write_to_silver(
    circuits_col_title_cap,
    silver_table,
    "t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "Ingestion_Timestamp",
        "Source_File",
        "batch_id"
    ],
)

In [0]:
%sql
select
  *
from
  formula1_incr.silver.circuits